# Mental Health in Tech Survey Analysis & Prediction
This notebook covers data exploration, cleaning, and training a Random Forest classifier to predict whether a tech employee seeks treatment for mental health conditions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

import warnings
warnings.filterwarnings('ignore')

### Load the Data

In [ ]:
df = pd.read_csv('../data/survey.csv')
df.head()

### Data Cleaning

In [ ]:
# Clean Age
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
df.loc[(df['Age'] < 18) | (df['Age'] > 75), 'Age'] = np.nan
df['Age'].fillna(df['Age'].median(), inplace=True)

# Clean Gender
df['Gender'] = df['Gender'].str.lower().str.strip()
male_str = ["male", "m", "male-ish", "maile", "mal", "male (cis)", "make", "male ", "man","msle", "mail", "malr","cis man", "cis male"]
female_str = ["cis female", "f", "female", "woman",  "femake", "female ","cis-female/femme", "female (cis)", "femail"]
trans_str = ["trans-female", "something kinda male?", "queer/she/they", "non-binary","nah", "all", "enby", "fluid", "genderqueer", "androgyne", "agender", "male leaning androgynous", "guy (-ish) ^_^", "trans woman", "neuter", "female (trans)", "queer", "ostensibly male, unsure what that really means"]

df['Gender'] = df['Gender'].replace(male_str, 'male')
df['Gender'] = df['Gender'].replace(female_str, 'female')
df['Gender'] = df['Gender'].replace(trans_str, 'other')
df['Gender'] = df['Gender'].replace(['a little about you', 'p'], 'other')

# Fill NAs
df['self_employed'].fillna('No', inplace=True)
df['work_interfere'].fillna('Don\'t know', inplace=True)


### Modeling

In [ ]:
features = ['Age', 'Gender', 'family_history', 'work_interfere', 'remote_work', 'tech_company', 'benefits', 'care_options', 'wellness_program', 'leave']
target = 'treatment'

X = df[features].copy()
y = df[target].copy()

encoders = {}
for col in X.columns:
    if X[col].dtype == 'object':
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
        encoders[col] = le
        
le_target = LabelEncoder()
y = le_target.fit_transform(y)
encoders['target'] = le_target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))